# Filter AMR papers with LLM

Use the `llm.py` helper to ask a model whether a paper's abstract asserts that a pathogen is showing **significant resistance** to a given antibiotic.


In [1]:

# --- LLM filtering configuration ---
import pandas as pd

from llm import create_llm_client
from amr.filtering import (
    DEFAULT_SYSTEM_PROMPT,
    prepare_pmids_to_review,
    run_llm_filter,
)

SYSTEM_PROMPT = """
You are an expert microbiologist reviewing evidence. Given a title and abstract, decide whether the authors explicitly state or conclude that the specified pathogen is showing clinically meaningful resistance to the specified antibiotic.

Rules:
- Answer "yes" only if the abstract clearly reports resistance, reduced susceptibility, treatment failure due to resistance, or emergence of resistant isolates for that pathogen/antibiotic pair.
- Answer "no" if the abstract does not make that claim, is about a different organism/drug, only mentions testing without resistance, or only provides background.
- Answer "uncertain" if the abstract is missing, ambiguous, or only indirectly related.
- Base the decision only on the provided text; do not infer beyond it.

Return compact JSON with keys: decision (yes/no/uncertain) and rationale (one short sentence).
"""

LLM_PROVIDER = "openai"
MODEL_NAME = "gpt-4o-mini"
NCBI_EMAIL = "your_email@example.com"
N_PMIDS_PER_RUN = 1000  # how many unprocessed PMIDs to handle this run
RATE_LIMIT = 0.35  # seconds between PubMed calls
OUTPUT_PATH = "pmids_results_filtered.parquet"
LOG_PATH = "pmid_filter.log"
MAX_LLM_CALLS = None  # set to an integer to cap spend per run
DRY_RUN = False  # set True to skip LLM calls
SAVE_EVERY = 50  # checkpoint frequency

llm_kwargs = {}
if MODEL_NAME:
    llm_kwargs["model"] = MODEL_NAME

llm = create_llm_client(provider=LLM_PROVIDER, **llm_kwargs)
print(f"Using provider={LLM_PROVIDER}, model={MODEL_NAME or 'provider default'}; dry_run={DRY_RUN}; max_llm_calls={MAX_LLM_CALLS}")


Using provider=openai, model=gpt-4o-mini; dry_run=False; max_llm_calls=None


In [2]:
#!pip install openai requests pandas lxml

In [3]:

import os
import pandas as pd
from amr.filtering import prepare_pmids_to_review

# Load previous decisions
if os.path.exists(OUTPUT_PATH):
    reviewed_df = pd.read_parquet(OUTPUT_PATH)
    reviewed = set(reviewed_df['pmid'].astype(str)) if not reviewed_df.empty else set()
    print(f"Resuming with {len(reviewed)} PMIDs already reviewed from {OUTPUT_PATH}")
else:
    reviewed_df = None
    reviewed = set()

pmid_df = pd.read_parquet("pmids_results.parquet")
# Exclude reviewed first, THEN cap to the next chunk
long_df = prepare_pmids_to_review(pmid_df, exclude_pmids=reviewed)

pmid_total = len(long_df)
reviewed_count = len(reviewed)
print(f"Loaded {pmid_total} candidate PMIDs after exclusion; reviewed set size: {reviewed_count}")

if N_PMIDS_PER_RUN:
    long_df = long_df.head(N_PMIDS_PER_RUN)

if len(long_df) == 0:
    print("No PMIDs remain after excluding reviewed. Consider using a fresh OUTPUT_PATH or adding more source data.")

print(f"PMIDs remaining to review this run: {len(long_df)}")
long_df.head()


Resuming with 7237 PMIDs already reviewed from pmids_results_filtered.parquet
Loaded 441907 candidate PMIDs after exclusion; reviewed set size: 7237
PMIDs remaining to review this run: 1000


,pmid,pathogen,antibiotic,start_date,end_date
0,23725464,Roseomonas mucosa,Amikacin,2010/01/01,2014/12/31
1,36548814,Rothia dentocariosa,Amikacin,2020/01/01,2024/12/31
2,35622725,Rothia dentocariosa,Amikacin,2020/01/01,2024/12/31
3,18287304,Rothia mucilaginosa,Amikacin,2005/01/01,2009/12/31
4,7329920,Salmonella enterica,Amikacin,1980/01/01,1984/12/31


In [4]:
%%time
filtered_df = run_llm_filter(
    long_df,
    llm,
    email=NCBI_EMAIL,
    rate_limit=RATE_LIMIT,
    save_every=SAVE_EVERY,
    system_prompt=SYSTEM_PROMPT,
    progress_every=10,
    resume_from=reviewed_df,
    seen_pmids=reviewed,
    output_path=OUTPUT_PATH,
    log_path=LOG_PATH,
    max_llm_calls=MAX_LLM_CALLS,
    dry_run=DRY_RUN,
)
filtered_df.head()


[1/1000] PMID 23725464 | Roseomonas mucosa vs Amikacin | interval 2010/01/01–2014/12/31
    decision=no | rationale=The abstract states the organism is generally susceptible to aminoglycosides, including amikacin.
[2/1000] PMID 36548814 | Rothia dentocariosa vs Amikacin | interval 2020/01/01–2024/12/31
    decision=no | rationale=The abstract does not mention resistance of Rothia dentocariosa to amikacin.
[3/1000] PMID 35622725 | Rothia dentocariosa vs Amikacin | interval 2020/01/01–2024/12/31
    decision=no | rationale=The abstract only mentions that amikacin effectively inhibits the growth of Rothia nasimurium without stating any resist
[4/1000] PMID 18287304 | Rothia mucilaginosa vs Amikacin | interval 2005/01/01–2009/12/31
    decision=yes | rationale=The isolate was multidrug-resistant and the patient had initial trials with amikacin.
[5/1000] PMID 7329920 | Salmonella enterica vs Amikacin | interval 1980/01/01–1984/12/31
    decision=no | rationale=The abstract does not report r

,pmid,decision,rationale,pathogen,antibiotic,title,abstract,start_date,end_date
0,27544979,no,The abstract discusses Achromobacter xylosoxid...,Achromobacter denitrificans,Spectinomycin,Properties of Achromobacter xylosoxidans highl...,We herein discovered a highly resistant clinic...,2015/01/01,2019/12/31
1,15004037,no,The abstract mentions resistance to spectinomy...,Acinetobacter baumannii,Spectinomycin,Molecular epidemiology of sequential outbreaks...,The molecular epidemiology of multidrug-resist...,2000/01/01,2004/12/31
2,19361944,no,The abstract does not explicitly state resista...,Acinetobacter baumannii,Spectinomycin,Genetic basis of resistance to aminoglycosides...,A total of 75 Acinetobacter isolates resistant...,2005/01/01,2009/12/31
3,19028622,no,The abstract does not mention Acinetobacter ba...,Acinetobacter baumannii,Spectinomycin,Nosocomial spread of armA-mediated high-level ...,"1,310 Enterobacteriaceae, 242 Pseudomonas aeru...",2005/01/01,2009/12/31
4,16000417,yes,"The abstract mentions gene aadA1, which confer...",Acinetobacter baumannii,Spectinomycin,Detection and typing of integrons in epidemic ...,Integrons were sought in Acinetobacter isolate...,2005/01/01,2009/12/31


In [5]:

print("Summary (this run):")

reviewed_pmids = set()
if 'reviewed_df' in globals() and reviewed_df is not None and not reviewed_df.empty:
    reviewed_pmids = set(reviewed_df['pmid'].astype(str))

new_df = filtered_df.copy()
if reviewed_pmids:
    new_df = new_df[~new_df['pmid'].astype(str).isin(reviewed_pmids)]

run_total = len(new_df)
run_yes = (new_df['decision'].str.lower() == 'yes').sum()
run_no = (new_df['decision'].str.lower() == 'no').sum()
run_unc = (new_df['decision'].str.lower() == 'uncertain').sum()
run_fraction = run_yes / run_total if run_total else 0
print(f"Processed this run: {run_total} abstracts; yes {run_yes}, no {run_no}, uncertain {run_unc} (yes fraction {run_fraction:.3f}).")

all_total = len(filtered_df)
all_yes = (filtered_df['decision'].str.lower() == 'yes').sum()
all_no = (filtered_df['decision'].str.lower() == 'no').sum()
all_unc = (filtered_df['decision'].str.lower() == 'uncertain').sum()
all_fraction = all_yes / all_total if all_total else 0
print(f"Cumulative: {all_total} abstracts; yes {all_yes}, no {all_no}, uncertain {all_unc} (yes fraction {all_fraction:.3f}).")

if os.path.exists(OUTPUT_PATH):
    tail = open(LOG_PATH, 'r', encoding='utf-8').read().splitlines()
    tail = tail[-10:] if tail else []
    print("
Log tail:")
    for line in tail:
        print(line)


Summary (this run):
Processed 8060 abstracts; positives 2008 (0.249).
Cumulative: 8060 abstracts; positives 2008 (0.249).
Log tail:
[973/1000] PMID 26182813 | Haemophilus influenzae vs Arbekacin | interval 2015/01/01–2019/12/31
    decision=no | rationale=The abstract does not mention resistance of Haemophilus influenzae to arbekacin.
[974/1000] PMID 25980052 | Haemophilus influenzae vs Arbekacin | interval 2015/01/01–2019/12/31
    decision=no | rationale=The abstract does not mention resistance of Haemophilus influenzae to arbekacin.
[975/1000] PMID 25980051 | Haemophilus influenzae vs Arbekacin | interval 2015/01/01–2019/12/31
    decision=no | rationale=The abstract does not report any resistance of Haemophilus influenzae to arbekacin.
[976/1000] PMID 25980050 | Haemophilus influenzae vs Arbekacin | interval 2015/01/01–2019/12/31
    decision=no | rationale=The abstract does not mention resistance of Haemophilus influenzae to arbekacin.
Finished: processed 8060 / 1000; positives 20

In [6]:

print(f"Saved results to {OUTPUT_PATH}")
filtered_df['decision'].value_counts()


Saved results to pmids_results_filtered.parquet


decision
no           5876
yes          2008
uncertain     176
Name: count, dtype: int64

In [7]:

# Build positive-only dataset and save
pmid_df_full = pd.read_parquet("pmids_results.parquet")

# Merge decisions onto exploded PMIDs
positive_flags = filtered_df[filtered_df["decision"].str.lower() == "yes"][["pmid"]].copy()
positive_rows = (
    pmid_df_full.explode("pmids")
    .assign(pmid=lambda d: d["pmids"].astype(str).str.extract(r"(\d+)")[0])
    .dropna(subset=["pmid"])
    .merge(positive_flags, on="pmid", how="inner")
)

# Reassemble to original grain
pmids_positive = (
    positive_rows.groupby(["pathogen", "antibiotic", "start_date", "end_date"], as_index=False)
    .agg(
        pmids=("pmid", list),
        count=("pmid", "size"),
        venue_histogram=("venue_histogram", "first"),
    )
)

positive_path = "pmids_results_positive.parquet"
pmids_positive.to_parquet(positive_path, index=False)
print(f"Saved positive-only dataset to {positive_path} with {len(pmids_positive)} rows")
pmids_positive.head()


Saved positive-only dataset to pmids_results_positive.parquet with 5568 rows


,pathogen,antibiotic,start_date,end_date,pmids,count,venue_histogram
0,Achromobacter denitrificans,Amikacin,1985/01/01,1989/12/31,[3163242],1,"{""Antimicrobial agents and chemotherapy"": 1}"
1,Achromobacter denitrificans,Amikacin,2005/01/01,2009/12/31,[19795625],1,"{""Mikrobiyoloji bulteni"": 1}"
2,Achromobacter denitrificans,Amikacin,2015/01/01,2019/12/31,[26935636],1,"{""Medicine"": 1, ""The Japanese journal of antib..."
3,Achromobacter denitrificans,Amoxicillin,2005/01/01,2009/12/31,[19795625],1,"{""Mikrobiyoloji bulteni"": 1, ""Scandinavian jou..."
4,Achromobacter denitrificans,Ampicillin,1985/01/01,1989/12/31,[3163242],1,"{""The Japanese journal of antibiotics"": 1, ""An..."


In [8]:
pmids_positive[pmids_positive['count']>50]

,pathogen,antibiotic,start_date,end_date,pmids,count,venue_histogram
58,Acinetobacter baumannii,Amikacin,2015/01/01,2019/12/31,"[32313533, 32032023, 31911917, 31706742, 31533...",82,"{""Annals of burns and fire disasters"": 2, ""Jou..."
59,Acinetobacter baumannii,Amikacin,2020/01/01,2024/12/31,"[39958989, 39766586, 39666200, 39418529, 39081...",78,"{""Frontiers in cellular and infection microbio..."
1575,Escherichia coli,Amikacin,2010/01/01,2014/12/31,"[25424607, 25379369, 25310915, 25285496, 25246...",67,"{""Journal of Nepal Health Research Council"": 2..."
1576,Escherichia coli,Amikacin,2015/01/01,2019/12/31,"[32045373, 31894059, 31893041, 31855054, 31683...",71,"{""Open access Macedonian journal of medical sc..."
1577,Escherichia coli,Amikacin,2020/01/01,2024/12/31,"[40673002, 39676194, 39619726, 39531253, 39518...",83,"{""JAC-antimicrobial resistance"": 8, ""Public he..."
2133,Klebsiella pneumoniae,Amikacin,2015/01/01,2019/12/31,"[32313533, 32058985, 31921701, 31855054, 31609...",52,"{""Annals of burns and fire disasters"": 1, ""Jou..."
2134,Klebsiella pneumoniae,Amikacin,2020/01/01,2024/12/31,"[39766529, 39734766, 39716087, 39678907, 39547...",90,"{""Archives of Razi Institute"": 5, ""Heliyon"": 3..."
3523,Pseudomonas aeruginosa,Amikacin,2010/01/01,2014/12/31,"[25429232, 25163250, 25141317, 24928215, 24899...",56,"{""Journal of preventive medicine and hygiene"":..."
3524,Pseudomonas aeruginosa,Amikacin,2015/01/01,2019/12/31,"[32313533, 31424553, 31262769, 31187452, 31123...",74,"{""Iranian journal of pharmaceutical research :..."
3525,Pseudomonas aeruginosa,Amikacin,2020/01/01,2024/12/31,"[39906216, 39766586, 39745212, 39659815, 39344...",94,"{""Archives of Razi Institute"": 5, ""Frontiers i..."
